# Milestone 4 — Transformer training (Experiments C and D)

**Runs on either Google Colab or Kaggle.** Full instructions, secrets and runtime
estimate: `research/scripts/MILESTONE_4_GPU_HANDOFF.md`.

Trains **mBERT (C)** and **XLM-R-base (D)** in-domain on Ax-to-Grind, 3 seeds each
(`{42, 123, 2026}`), writes metrics through the project's existing metrics contract,
and pushes each checkpoint to a **staging** HF repo.

One notebook, both platforms — not two copies that drift. The only differences are
the secret store, the working directory and how the metrics zip comes back; all
three live in `research/src/notebook_env.py`. The training itself is identical.

## Before you start

### On Kaggle

1. **Settings → Accelerator → GPU T4 x2** (or P100). Kaggle has no
   `Runtime → Change runtime type` menu — the accelerator is a notebook setting.
2. **Settings → Internet → On.** ⚠️ Kaggle disables internet **by default**, and
   with it off the repo clone, `pip install` and the HF checkpoint push all fail.
   This is the easiest blocker to miss. (Requires a phone-verified Kaggle account.)
3. **Add-ons → Secrets** → add `HF_TOKEN` (your HF **write** token) and make sure
   it is **attached to this notebook**.
4. Set `HF_STAGING_PREFIX` in the config cell below to your HF username.

### On Colab

1. **Runtime → Change runtime type → T4 GPU**
2. Add your HF **write** token to Colab **Secrets** (🔑 sidebar) as `HF_TOKEN`,
   with notebook access on.
3. Set `HF_STAGING_PREFIX` in the config cell below to your HF username.

Colab has internet on always; there is no toggle to set.

## 1. Pre-flight — GPU and internet

Stops immediately if there is no GPU, rather than silently spending hours on CPU.

Also checks outbound internet, because on **Kaggle it is off by default**. Without
it the clone, the `pip install` and the checkpoint push all fail — several cells
apart, with three unrelated-looking errors. Checking here makes it one message.

In [ ]:
import os, socket, subprocess, sys

# Which host is this? This cell runs BEFORE the repo is cloned, so it cannot import
# research/src/notebook_env.py yet — this is the one place the detection is
# duplicated, deliberately, and only the wording of messages depends on it.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle")

# --- GPU -------------------------------------------------------------------
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    sys.exit(
        "No GPU detected. "
        + (
            "Settings -> Accelerator -> GPU T4 x2 (or P100), then rerun."
            if ON_KAGGLE
            else "Runtime -> Change runtime type -> T4 GPU, then rerun."
        )
    )
print(out.stdout)

# --- Internet --------------------------------------------------------------
# Kaggle notebooks have internet DISABLED by default; git clone, pip install and
# the HF push all need it.
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
    print("internet: OK")
except OSError as exc:
    sys.exit(
        f"No outbound internet ({exc}).\n"
        + (
            "On Kaggle this is the DEFAULT. Settings -> Internet -> On (right-hand "
            "panel; needs a phone-verified account), then rerun from the top."
            if ON_KAGGLE
            else "Colab normally has internet — check the connection and rerun."
        )
    )

# --- Pin to ONE GPU --------------------------------------------------------
# Kaggle's default accelerator is "GPU T4 x2". With two GPUs visible, the HF
# Trainer transparently wraps the model in nn.DataParallel and treats
# per_device_train_batch_size as PER DEVICE -- so the configured 16 becomes an
# effective batch of 32, and the 401 optimizer steps/epoch this project's runtime
# estimate and 3-seed design are built on become ~201. That is a silent change to
# the training dynamics, not a speed-up.
#
# The research design is frozen (config unchanged, batch size unchanged); making
# only one GPU visible is what KEEPS it identical to the single-T4 Colab run this
# was written for. Set before torch is imported anywhere, and inherited by the
# `!python -m ...` training subprocesses below.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES=0 (single GPU, matching the configured batch size)")

print("platform:", "Kaggle" if ON_KAGGLE else "Colab / other")

## 2. Configuration

`HF_STAGING_PREFIX` is not secret — set it here. `HF_TOKEN` is read from the
platform's secret store (Colab Secrets / Kaggle Add-ons → Secrets) in section 3,
once the repo that knows how to read it is present.

In [ ]:
# ---- EDIT THIS ----
HF_STAGING_PREFIX = "your-hf-username"   # e.g. "rehman-ayoub"
REPO_URL = "https://github.com/<your-github-username>/<repo-name>.git"
REPO_BRANCH = "main"
# -------------------

# Where the clone goes. /content exists only on Colab; on Kaggle the writable,
# roomy, session-persisted directory is /kaggle/working. Once the repo is synced,
# research/src/notebook_env.py owns this mapping (repo_dir()) — it cannot be
# imported yet, because it lives in the repo about to be cloned.
REPO_DIR = "/kaggle/working/repo" if ON_KAGGLE else "/content/repo"

# Identifies THIS notebook document. Bumped whenever a cell below changes, and
# checked against the committed copy after the repo is synced (section 3).
# Restarting the runtime restarts the kernel but does NOT reload the notebook
# source, so an open tab can quietly keep running pre-fix cells against a freshly
# updated repo — which is exactly how the 2026-08-15 run failed. This turns that
# into a clear error. Applies equally to Kaggle's "Factory reset".
NOTEBOOK_REVISION = 8

assert HF_STAGING_PREFIX != "your-hf-username", "Set HF_STAGING_PREFIX to your HF username first."
print("staging namespace:", HF_STAGING_PREFIX)
print("repo directory:   ", REPO_DIR)
print("notebook revision:", NOTEBOOK_REVISION)

## 3. Sync the repo, read the secret, install pinned dependencies

The first cell **re-syncs to the branch tip every run**, not only on a first clone,
and prints the commit it landed on. The second checks that the notebook you are
running is not older than the one in that commit — restarting the runtime reloads
the kernel but *not* the notebook source, so an open tab can otherwise keep
executing pre-fix cells against fixed repo code.

The third reads `HF_TOKEN`, and comes **after** the sync on purpose: it uses
`research/src/notebook_env.py`, which lives in the repo and is what lets this one
notebook run on both platforms. That module is stdlib-only, so it imports fine
here — before `pip install` has run.

In [ ]:
import os, subprocess, sys

# Sync to the branch tip on EVERY run, not just when the directory is missing.
#
# The previous version cloned only if the directory was absent. Restarting the
# session keeps the working directory, so on any rerun the clone was skipped and the
# repo silently stayed at whatever commit it was first cloned at — no output said so.
# Two sources of truth (notebook cells vs repo code) then drift apart with nothing
# reporting it. `checkout -B` resets the local branch onto origin's tip, which is
# idempotent and safe here: research/data/raw/ is gitignored, and untracked files
# (downloaded corpora, produced metrics) are left alone by a hard branch reset.
#
# Plain git throughout. Nothing in this cell is platform-specific beyond REPO_DIR,
# so it behaves identically on Colab and Kaggle — but Kaggle needs internet ENABLED
# for the clone and fetch to work at all (checked in the pre-flight cell).
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin", REPO_BRANCH], check=True)
subprocess.run(
    ["git", "checkout", "--quiet", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True
)

# `import research.src...` needs the repo root on sys.path. Colab's IPython puts the
# CWD there implicitly; Kaggle's starts in /kaggle/working and does not.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo synced to:")
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

In [ ]:
# Stale-notebook guard. Compares the notebook YOU are running against the copy just
# synced from the repo, and stops if yours is older.
#
# Why this exists: on 2026-08-15 a run had a correctly-updated repo but a pre-fix
# notebook tab. The cells issued old commands against new code, so the data check ran
# unscoped and the smoke test hit the torchvision error the new cells prevent. Nothing
# reported the mismatch; it looked like the fixes had simply not worked.
import json, re

with open("research/notebooks/04_transformer_training.ipynb", encoding="utf-8") as fh:
    committed = json.load(fh)

marker = re.compile(r"NOTEBOOK" + r"_REVISION\s*=\s*(\d+)")
found = [
    int(m.group(1))
    for cell in committed["cells"]
    for m in marker.finditer("".join(cell["source"]))
]
repo_revision = max(found) if found else 0

if repo_revision > NOTEBOOK_REVISION:
    raise SystemExit(
        f"STALE NOTEBOOK — you are running revision {NOTEBOOK_REVISION}, the repo has "
        f"revision {repo_revision}.\n\n"
        "Restarting the runtime restarts the kernel; it does NOT reload the notebook "
        "source in an already-open tab.\n"
        "Fix: close this tab, re-open the notebook from GitHub "
        "(File -> Open notebook -> GitHub tab -> this repo), and run from the top."
    )

print(f"notebook revision {NOTEBOOK_REVISION}, repo revision {repo_revision} — in sync")

In [ ]:
import os

# Read the write token from the platform's secret store, so it never appears in
# saved notebook output. notebook_env picks the right one:
#   Colab  -> google.colab.userdata            (Secrets, key icon, left sidebar)
#   Kaggle -> kaggle_secrets.UserSecretsClient (Add-ons -> Secrets)
# and raises with the platform-appropriate fix if the secret is missing, empty, or
# the host is neither. Unit-tested in research/tests/test_notebook_env.py.
#
# Imported before the pip install below, which is safe: notebook_env is stdlib-only.
from research.src.notebook_env import detect_platform, get_secret, platform_markers

# Print the EVIDENCE, not just the verdict. On 2026-08-19 this cell reported
# "colab" on a real Kaggle P100 -- google.colab is importable there, because
# Kaggle's image is built FROM the Colab image -- and there was nothing to
# inspect but the wrong answer. Detection is now Kaggle-first; this makes any
# future misdetection diagnosable from the notebook output alone.
print("detected platform:", detect_platform())
for _platform, _found in platform_markers().items():
    print(f"  {_platform:<7} signals: {_found or '(none)'}")

os.environ["HF_TOKEN"] = get_secret("HF_TOKEN")
os.environ["HF_STAGING_PREFIX"] = HF_STAGING_PREFIX

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("HF_STAGING_PREFIX set:", os.environ["HF_STAGING_PREFIX"])

In [ ]:
# Step 1 of 3 — remove the image's preinstalled torchvision/torchaudio BEFORE
# installing. Applies to Kaggle as much as to Colab: Kaggle's notebook image is
# built FROM the Colab runtime image, so it ships the same preinstalled copies.
#
# They are compiled against whatever torch the image shipped with. The install below
# moves torch to this project's pin, and the leftovers then register C++ ops against
# the wrong ABI, so `from transformers import Trainer` dies with
#     RuntimeError: operator torchvision::nms does not exist
# transformers guards that import with is_torchvision_available(), which only checks
# whether the package is INSTALLED, not whether it imports — a present-but-broken
# copy passes the guard and raises RuntimeError, which nothing on that path catches.
# With torchvision absent the guard is simply False and the block is skipped.
# This project does zero vision and zero audio work. See research/requirements.txt.
#
# pip will warn that fastai/timm now have an unsatisfied torchvision requirement.
# That is expected and harmless — nothing in this pipeline imports them.
!pip uninstall -y -q torchvision torchaudio

# Step 2 of 3 — install the pinned set (REPRODUCIBILITY.md Section 1). On Linux
# (both platforms are Linux) the plain torch pin resolves to the CUDA build,
# which is what a T4 or a P100 needs.
!pip install -q -r research/requirements.txt

# Step 3 of 3 — rewrite numpy and scipy COMPLETELY, over the top of the image's
# copies. Same class of problem as the torchvision block above (a preinstalled
# package left in an inconsistent state), but a different mechanism needing a
# different remedy — and it is NOT a scipy/numpy version conflict, despite the
# traceback looking exactly like one. See DECISION_REGISTER.md M4-4.
#
# What actually failed on Kaggle on 2026-08-19, at `from transformers import Trainer`:
#
#     ImportError: cannot import name '_center' from 'numpy._core.umath'
#                  (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)
#
# That import is INSIDE numpy, not at a scipy/numpy boundary. numpy/_core/umath.py
# is a pure-Python shim that re-exports from the COMPILED _multiarray_umath
# extension, and `_center` is a ufunc living in that binary.
#
# CORRECTED by M4-5: the mixture is in MEMORY, not on disk. The image imports numpy
# at kernel boot; pip then correctly replaces it on disk; the kernel keeps the old
# module objects. The lazy numpy.char/_core.strings then load from the NEW files
# against the OLD cached umath. Both pip and the kernel are right at once — which is
# why the 2026-08-19 run reported numpy 2.0.2 while pip reported 2.5.2 installed.
# The environment gate below therefore runs in a subprocess, where no stale modules
# exist. This force-reinstall is KEPT as cheap insurance against a genuinely partial
# on-disk install, which would look identical from inside the kernel.
#
# scipy is only the messenger. Plain `import numpy` does NOT load the affected
# submodules — numpy.char and numpy._core.strings are lazy. `from numpy import *`
# DOES, and scipy's array_api_compat shim is the first thing in the process to run
# it, which is why every earlier cell, and torch itself, imported fine.
#
# scipy is therefore NOT uninstalled the way torchvision is: it is genuinely
# required. `import sklearn.metrics` — the path every metrics file this project
# writes goes through — pulls in 493 scipy modules, and scikit-learn declares scipy
# as a hard dependency. Removing it would break the run outright.
#
# --force-reinstall rewrites every file of both packages instead of skipping them as
# "already satisfied", which is what repairs the mixed install; --no-deps keeps it
# surgical, so nothing else in the resolved set is disturbed.
!pip install -q --force-reinstall --no-deps numpy==2.5.2 scipy==1.18.0

In [ ]:
# Environment gate — runs in a FRESH SUBPROCESS, not in this kernel. That is the
# fix for DECISION_REGISTER.md M4-5, not a stylistic preference.
#
# Kaggle and Colab import numpy at kernel boot. The dependency cell above then
# replaces numpy on disk, but this kernel keeps the module objects it already
# holds. numpy.char and numpy._core.strings are LAZY, so the first
# `from numpy import *` after the install reads those two files from the NEW numpy
# while numpy._core.umath is still the OLD cached one, and the new strings.py asks
# for a `_center` ufunc the old compiled extension does not have:
#
#     ImportError: cannot import name '_center' from 'numpy._core.umath'
#
# pip and the kernel are BOTH right at the same time, which is why the 2026-08-19
# run reported numpy 2.0.2 (the image's, held in memory) while pip correctly
# reported 2.5.2 installed. Reproduced locally; there is no shadow install and no
# corrupt install on disk.
#
# Every real step below already runs as its own `python -m ...` process, so all of
# them read the freshly installed packages and were never affected. Running the
# gate the same way makes it check the environment the TRAINING actually uses,
# instead of this kernel's stale view of it — and removes any need to restart the
# session mid-notebook.
#
# subprocess.run + an explicit raise, rather than a bare `!python`: a `!` command's
# non-zero exit does NOT stop "Run All", so a failed gate would otherwise scroll by
# and the smoke test would run anyway.
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "research/scripts/check_gpu_env.py"],
    cwd=REPO_DIR,
)

if result.returncode != 0:
    raise SystemExit(
        "Environment gate FAILED — see the checks above. Do not continue to the "
        "smoke test; fix the reported items first."
    )

## 4. Data check — Ax-to-Grind only

Training reads the **committed split index files**, so the raw corpus must be
present. `research/data/raw/` is gitignored, so this re-downloads it (no
credentials needed).

Scoped to Ax-to-Grind deliberately: Experiments C and D train and evaluate on it
alone. Notri-Fact is the held-out **cross-dataset** test set and is not read until
Milestone 5 (`EXPERIMENT_PLAN.md` step 4), so requiring it here would fail the run
over a file this milestone never opens. The full-coverage checks stay exactly as
they are for CI and the complete pipeline — this cell selects from them, it does
not replace them.

In [ ]:
!python -m research.src.data.download --only ax_to_grind

# download.py regenerates MANIFEST.sha256 wholesale from whatever is on disk, so
# after a single-dataset download it holds hashes recomputed from the files just
# fetched — verifying those files against it would be circular and would prove
# nothing. Restore the COMMITTED manifest, which is the actual dataset-version
# anchor (REPRODUCIBILITY.md Section 3), before checking anything against it.
!git checkout -- research/data/raw/MANIFEST.sha256

# Presence + checksum, Ax-to-Grind only. `-k ax_to_grind` selects
# test_manifest_covers_ax_to_grind plus the three per-file checksum cases; the
# Notri-Fact and whole-manifest cases stay in the file untouched for CI.
!python -m pytest research/tests/test_raw_data_integrity.py -k ax_to_grind -q

# Schema, label and row-count validation, Ax-to-Grind only.
!python -m research.src.data.validate --only ax_to_grind

## 5. Smoke test — a few real steps before the real job

**If this fails, stop and send the error.** It costs a minute and catches setup
problems before an hour of GPU time is spent on them.

In [ ]:
!python -m research.src.models.transformer --dry-run --experiments D --dry-run-max-length 128

## 5b. RECOVERY MODE — re-score existing checkpoints instead of retraining

**Skip this section entirely on a normal run.** It exists for one situation
(`DECISION_REGISTER.md` M4-6): the six training runs completed and pushed their
checkpoints, but the metrics files were lost with the Kaggle session's working
directory before they could be downloaded.

Re-scoring is not a shortcut, it is the *faithful* recovery. `load_best_model_at_end`
means the checkpoint on the Hub is the exact model that produced the lost numbers,
and scoring is `argmax` over logits with no sampling — so it reproduces them. A
retrain would not: GPU training is not bit-deterministic, so it would produce
different weights and overwrite the seed branches whose revision SHAs are already
recorded.

Two things it cannot recover, which are written as `null` rather than invented:
`train_runtime_seconds` and `train_loss`. Neither feeds RQ1, RQ2 or RQ3.

Run **Step 0 first** and read its output. If it reports any seed as missing or
incomplete, only those seeds need retraining — everything it lists as OK can be
recovered in minutes.

In [ ]:
# RECOVERY STEP 0 — what is actually on the Hub? CPU only, seconds, no GPU cost.
#
# Checks all six seed branches across both staging repos and requires config,
# weights AND tokenizer on each: a branch with weights but no tokenizer cannot be
# re-scored, which a bare "does the branch exist" check would miss.
#
# Read this output before running Step 1. Exit 0 = everything is recoverable.
import subprocess
import sys

step0 = subprocess.run(
    [sys.executable, "research/scripts/inventory_staging_checkpoints.py"],
    cwd=REPO_DIR,
)
print()
if step0.returncode == 0:
    print("STEP 0 PASSED — every checkpoint is present. Run Step 1 below.")
else:
    print(
        "STEP 0 FOUND PROBLEMS — read the list above. Seeds reported OK can still be\n"
        "recovered by Step 1; only the ones named as missing need retraining."
    )

In [ ]:
# RECOVERY STEP 1 — re-score the pushed checkpoints. No training.
#
# ~2,750 rows per seed (val 1,372 + test 1,374), forward-only, so the compute is
# under a minute per seed; the wall-clock is dominated by downloading ~5.4 GB of
# checkpoints. Each seed's metrics are uploaded to the results repo as soon as they
# are written (M4-6), so an interruption here cannot lose completed work the way
# the original run did.
#
# To recover only some seeds (if Step 0 reported others missing), add e.g.
#     --experiments D --seeds 42 123
import subprocess
import sys

recovery = subprocess.run(
    [sys.executable, "-m", "research.src.models.evaluate_checkpoint"],
    cwd=REPO_DIR,
)

if recovery.returncode != 0:
    raise SystemExit(
        "Recovery did not complete cleanly — see the output above. Metrics that were "
        "written still exist locally, but at least one did not reach the Hub."
    )
print("\nRecovery complete. Run the summary and packaging cells below.")

## 6. The real runs — 2 models × 3 seeds

~40–80 min total. Checkpoints push to staging as each seed finishes, so an
interrupted session does not lose completed work.

If you would rather split across two sessions, run the two cells separately.

In [ ]:
!python -m research.src.experiments.run_in_domain --models transformer --experiments C

In [ ]:
!python -m research.src.experiments.run_in_domain --models transformer --experiments D

## 7. Summary — copy this table back

In [ ]:
import glob
import json
import statistics

rows = []
for path in sorted(glob.glob("research/results/metrics/[CD]_*.json")):
    d = json.load(open(path, encoding="utf-8"))
    meta = d["run_metadata"]

    # train_runtime_seconds is None for a file recovered by re-scoring a checkpoint
    # (M4-6): it is not observable from a checkpoint, so it is recorded as null
    # rather than invented. `.get(key, 0)` does NOT help here -- the key EXISTS and
    # holds None, so the default never applies and `None / 60` would raise.
    runtime = meta.get("train_runtime_seconds")
    minutes = round(runtime / 60, 1) if runtime is not None else "n/a"

    rows.append((
        d["experiment_id"], d["model"], d["split"], d["seed"],
        round(d["metrics"]["macro_f1"], 4),
        round(d["metrics"]["accuracy"], 4),
        d["prediction_collapse"]["is_collapsed"],
        meta.get("truncation", {}).get("pct_truncated"),
        minutes,
        # Marks a re-scored file so a reader never mistakes it for a training run.
        "yes" if meta.get("evaluation_only_recovery", {}).get("recovered") else "",
    ))

hdr = (
    f"{'exp':<4}{'model':<34}{'split':<6}{'seed':<7}{'macroF1':>9}{'acc':>8}"
    f"{'collapsed':>11}{'trunc%':>8}{'min':>7}{'recov':>7}"
)
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(
        f"{r[0]:<4}{r[1]:<34}{r[2]:<6}{r[3]:<7}{r[4]:>9}{r[5]:>8}"
        f"{str(r[6]):>11}{str(r[7]):>8}{str(r[8]):>7}{r[9]:>7}"
    )

if any(r[9] for r in rows):
    print(
        "\nNOTE: rows marked 'recov' were re-scored from their pushed checkpoints "
        "(DECISION_REGISTER.md M4-6).\n"
        "Their metrics are exact -- load_best_model_at_end makes the pushed "
        "checkpoint the model that\nproduced them -- but train_runtime_seconds and "
        "train_loss are unavailable and recorded as null."
    )

print("\nSeed variance (test macro-F1) — report this, EXPERIMENT_PLAN.md Section 5:")
for exp in ("C", "D"):
    vals = [r[4] for r in rows if r[0] == exp and r[2] == "test"]
    if len(vals) > 1:
        print(f"  {exp}: mean={statistics.mean(vals):.4f} sd={statistics.stdev(vals):.4f} values={vals}")

## 8. Package the metrics — **the run is not done until these are on the Hub**

`DECISION_REGISTER.md` M4-6. This cell pushes the metrics to Hugging Face **first**,
verifies they actually arrived, and only then produces a zip for convenience. It
stops the notebook if the upload cannot be confirmed.

That ordering is deliberate. Milestone 4's first run packaged a zip, the Output-tab
download silently failed, and the Kaggle session's working directory was gone before
it could be retried — every metrics file lost, while all six checkpoints survived
because they were pushed as each seed finished. The Output tab is a convenience, not
a copy.

Also bring back the staging repo IDs and **revision SHAs** printed above
(`REPRODUCIBILITY.md` Section 6 needs repo ID *and* revision — a branch name moves, a
SHA does not).

In [ ]:
import sys
import zipfile
from pathlib import Path

sys.path.insert(0, REPO_DIR)
from research.src.evaluation.results_push import (  # noqa: E402
    push_result_files,
    verify_results_uploaded,
)
from research.src.notebook_env import deliver_file, working_root  # noqa: E402

metrics_dir = Path(REPO_DIR) / "research" / "results" / "metrics"
metrics = sorted(metrics_dir.glob("[CD]_*.json"))
assert metrics, "no C/D metrics files found — did the training cells actually run?"

# --- 1. DURABILITY FIRST (M4-6) ------------------------------------------------
# The Hub upload happens BEFORE the zip and before any thought of downloading.
#
# Milestone 4's first run did the opposite: it packaged a zip, the Output-tab
# download silently failed while Kaggle was returning 503s, and the session's
# working directory was gone before it could be retried. All six checkpoints
# survived, because they were pushed as each seed finished; every metrics file was
# lost, because they were only ever on that disk. A Kaggle draft session's working
# directory is not a safe place to leave the only copy of a milestone's results,
# even for a few minutes.
#
# Per-seed pushes already happened during training/recovery; this is the sweep that
# catches anything that failed a retry back then.
print("pushing metrics to the Hub...")
push_status = push_result_files(metrics)
print(f"  {push_status}")

# --- 2. VERIFY IT ACTUALLY LANDED ----------------------------------------------
# Asking the Hub what it has, rather than trusting the upload call's return value.
# verify_results_uploaded fails closed: an unreachable Hub reads as unverified.
verification = verify_results_uploaded([p.name for p in metrics])
print(f"  verification: {verification}")

if not verification["verified"]:
    raise SystemExit(
        "RESULTS ARE NOT SAFE YET — "
        f"{len(verification.get('missing', []))} file(s) are not on the Hub: "
        f"{verification.get('missing')}\n"
        f"reason: {verification.get('reason', 'see above')}\n\n"
        "Do NOT close this session. Re-run this cell. The metrics still exist in\n"
        f"{metrics_dir}, but only there — that is exactly the state M4-6 exists to\n"
        "prevent. If the Hub is down, copy the JSON out by hand before stopping."
    )

print(f"\nAll {len(metrics)} metrics files are on the Hub: {verification['repo_id']}")
print("The results are now durable. Everything below is convenience.")

# --- 3. Convenience copy for the Output tab ------------------------------------
archive = working_root() / "milestone4_metrics.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in metrics:
        zf.write(path, arcname=path.name)
print(f"\nzipped {len(metrics)} files -> {archive}")
print(deliver_file(archive))